# Mount Google Drive

This notebook trains a GRU-based intent classifier for the AI Loan Advisory Chatbot.
The trained model and preprocessing files will be stored in the project directory so they can be loaded directly by the RAG pipeline in the next notebook.

In [1]:
# Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# Import Required Libraries

The intent classifier uses PyTorch for model training and common Python libraries for data handling, preprocessing, evaluation, and saving artifacts. All imports are kept together so the notebook remains organized and easy to maintain.

In [2]:
# Standard libraries
import os
import json
import pickle
import random
from collections import Counter

# Numerical computing
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# Progress bar
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cpu


# Project Configuration

All project paths and training settings are defined in one place. Keeping these values centralized makes the notebook easier to update and ensures the saved model and preprocessing files can be loaded directly in the next stage of the project.

In [3]:
# -----------------------------
# Project Paths
# -----------------------------

PROJECT_ROOT = "/content/drive/MyDrive/AI_Loan_Advisory_Chatbot"

PROCESSED_DIR = os.path.join(PROJECT_ROOT, "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")

# Create required directories
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

# -----------------------------
# Model Save Paths
# -----------------------------

MODEL_PATH = os.path.join(MODELS_DIR, "gru_intent_model.pth")
VOCAB_PATH = os.path.join(MODELS_DIR, "vocabulary.pkl")
LABEL_ENCODER_PATH = os.path.join(MODELS_DIR, "label_encoder.pkl")
TOKENIZER_PATH = os.path.join(MODELS_DIR, "tokenizer.pkl")

# -----------------------------
# Dataset Path
# -----------------------------

INTENT_DATASET_PATH = os.path.join(DATASET_DIR, "intent_dataset.json")

# -----------------------------
# Training Configuration
# -----------------------------

MAX_SEQUENCE_LENGTH = 20
MIN_WORD_FREQUENCY = 1

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20

# -----------------------------
# Display Configuration
# -----------------------------

print("=" * 60)
print("Project Configuration")
print("=" * 60)
print(f"Project Root      : {PROJECT_ROOT}")
print(f"Dataset Directory : {DATASET_DIR}")
print(f"Models Directory  : {MODELS_DIR}")
print(f"Device            : {DEVICE}")
print(f"Batch Size        : {BATCH_SIZE}")
print(f"Epochs            : {NUM_EPOCHS}")
print(f"Learning Rate     : {LEARNING_RATE}")
print("=" * 60)

Project Configuration
Project Root      : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot
Dataset Directory : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/dataset
Models Directory  : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/models
Device            : cpu
Batch Size        : 32
Epochs            : 20
Learning Rate     : 0.001


# Define Intent Labels

The chatbot is designed to recognize common loan-related queries along with a few general conversational intents. These labels will be used throughout the notebook for dataset creation, model training, and evaluation.

In [4]:
# Supported intent labels

INTENTS = [
    "loan_eligibility",
    "interest_rate",
    "emi_calculation",
    "required_documents",
    "processing_time",
    "loan_features",
    "loan_comparison",
    "repayment",
    "charges_fees",
    "apply_loan",
    "home_loan",
    "personal_loan",
    "education_loan",
    "gold_loan",
    "vehicle_loan",
    "loan_against_property",
    "greeting",
    "general_query",
    "unknown"
]

print(f"Total Supported Intents: {len(INTENTS)}\n")

for idx, intent in enumerate(INTENTS, start=1):
    print(f"{idx:2}. {intent}")

Total Supported Intents: 19

 1. loan_eligibility
 2. interest_rate
 3. emi_calculation
 4. required_documents
 5. processing_time
 6. loan_features
 7. loan_comparison
 8. repayment
 9. charges_fees
10. apply_loan
11. home_loan
12. personal_loan
13. education_loan
14. gold_loan
15. vehicle_loan
16. loan_against_property
17. greeting
18. general_query
19. unknown


# Create Intent Dataset

A supervised classifier requires labeled examples for each intent. This section creates a structured dataset of sample user queries that can be expanded later without changing the training pipeline.

In [5]:
# Intent dataset
intent_dataset = {
    "greeting": [
        "Hello",
        "Hi",
        "Good morning",
        "Good evening",
        "Hey",
        "Greetings",
        "Hi there",
        "Hello chatbot",
        "Can you help me?",
        "Anyone there?"
    ],

    "loan_eligibility": [
        "Am I eligible for a home loan?",
        "Who can apply for a personal loan?",
        "What is the eligibility criteria?",
        "Can I get a loan with my salary?",
        "Minimum income required for loan?",
        "Who qualifies for an education loan?",
        "Can self employed people apply?",
        "Is my age eligible for a loan?",
        "Loan eligibility requirements",
        "Can I apply for this loan?"
    ],

    "interest_rate": [
        "What is the interest rate?",
        "Current home loan interest rate",
        "Interest rate for SBI personal loan",
        "Education loan interest",
        "Gold loan interest rate",
        "Vehicle loan interest",
        "How much interest will I pay?",
        "Latest loan interest",
        "Fixed or floating interest?",
        "Tell me the interest charges",
        "What is the interest rate for SBI Home Loan?",
"What is the interest rate for HDFC Home Loan?",
"What is the home loan interest rate?",
"What is the current home loan interest?",
"Current SBI Home Loan interest rate",
"Current HDFC Home Loan interest rate",
"What interest does SBI charge for home loans?",
"What interest does HDFC charge for home loans?",
"Tell me the home loan interest rate.",
"Home loan rate of interest",
"SBI Home Loan ROI",
"HDFC Home Loan ROI",
"Latest SBI Home Loan interest",
"Latest HDFC Home Loan interest",
"Floating interest rate for home loan",
"Home loan floating rate",
"Current housing loan interest",
"What is today's home loan rate?",
"Interest charged on home loan",
"Rate of interest on home loan",
"Personal loan interest rate SBI",
"Personal loan interest rate HDFC",
"What is SBI Personal Loan interest?",
"What is HDFC Personal Loan interest?",
"Gold loan interest rate SBI",
"Gold loan interest rate",
"Vehicle loan interest rate",
"Car loan interest rate",
"Education loan interest rate",
"Loan Against Property interest rate",
"What is the ROI for education loan?",
"What is the ROI for car loan?",
"What is the ROI for personal loan?",
"How much interest is charged on education loan?",
"Tell me the interest rate for gold loan.",
"What is the latest loan interest?",
"Current bank loan interest rate",
"Current lending rate",
"What is the annual interest rate?",
"Interest percentage for loan",
"How much interest will I pay?",
"Interest applicable on loan",
"What is the applicable interest rate?",
"Tell me the current ROI.",
"Interest on SBI housing loan"
    ],

    "emi_calculation": [
        "Calculate my EMI",
        "How much EMI will I pay?",
        "Monthly installment",
        "EMI for 10 lakh loan",
        "Loan EMI calculation",
        "Estimate EMI",
        "EMI details",
        "Monthly payment amount",
        "Show EMI",
        "Loan repayment per month"
    ],

    "required_documents": [
        "What documents are required?",
        "Documents needed for home loan",
        "Required papers",
        "Income proof required?",
        "Identity proof needed?",
        "Which documents should I submit?",
        "Document checklist",
        "Loan application documents",
        "KYC documents",
        "Required certificates"
    ],

    "processing_time": [
        "How long does approval take?",
        "Loan processing time",
        "When will my loan be approved?",
        "Approval duration",
        "Time required for sanction",
        "Loan verification time",
        "Processing period",
        "Expected approval date",
        "How many days for approval?",
        "Loan timeline"
    ],

    "loan_features": [
        "Tell me loan features",
        "Benefits of this loan",
        "Loan advantages",
        "Key features",
        "Special features",
        "Why should I choose this loan?",
        "Loan highlights",
        "Available benefits",
        "Loan facilities",
        "Explain loan features"
    ],

    "loan_comparison": [
        "Compare SBI and HDFC home loans",
        "Best loan option",
        "Which bank offers lower interest?",
        "Compare personal loans",
        "Difference between loans",
        "Loan comparison",
        "Best education loan",
        "Which loan is better?",
        "Compare two loans",
        "Loan comparison details"
    ],

    "repayment": [
        "Repayment options",
        "Loan repayment schedule",
        "Can I prepay?",
        "Foreclosure charges",
        "Repayment tenure",
        "EMI repayment",
        "Loan closure",
        "Repay my loan",
        "Early repayment",
        "Installment schedule"
    ],

    "charges_fees": [
        "Processing fee",
        "Hidden charges",
        "Loan fees",
        "Penalty charges",
        "Documentation fee",
        "Service charges",
        "Loan expenses",
        "Application fee",
        "Additional charges",
        "Total fees"
    ],

    "apply_loan": [
        "How do I apply?",
        "Loan application process",
        "Apply for loan",
        "Online application",
        "Start my application",
        "Application steps",
        "Submit loan request",
        "Apply now",
        "Loan registration",
        "Application procedure"
    ],

    "home_loan": [
        "Tell me about home loan",
        "Housing loan",
        "Home finance",
        "SBI home loan",
        "HDFC home loan",
        "House purchase loan",
        "Loan for buying house",
        "Home loan details",
        "House loan eligibility",
        "Property purchase loan"
    ],

    "personal_loan": [
        "Personal loan",
        "Need a personal loan",
        "Personal finance loan",
        "Quick personal loan",
        "Personal loan details",
        "SBI personal loan",
        "HDFC personal loan",
        "Unsecured personal loan",
        "Personal loan eligibility",
        "Personal loan features"
    ],

    "education_loan": [
        "Education loan",
        "Student loan",
        "Loan for studies",
        "Higher education loan",
        "Study abroad loan",
        "College loan",
        "Education finance",
        "Student education loan",
        "Loan for university",
        "Education loan details"
    ],

    "gold_loan": [
        "Gold loan",
        "Loan against gold",
        "Gold jewellery loan",
        "Pledge gold",
        "Gold finance",
        "Gold loan interest",
        "Gold loan eligibility",
        "Gold loan details",
        "Need gold loan",
        "Gold backed loan"
    ],

    "vehicle_loan": [
        "Vehicle loan",
        "Car loan",
        "Bike loan",
        "Auto loan",
        "Finance my car",
        "Purchase vehicle loan",
        "New car loan",
        "Vehicle finance",
        "Car loan eligibility",
        "Vehicle loan interest"
    ],

    "loan_against_property": [
        "Loan against property",
        "Property mortgage loan",
        "Mortgage loan",
        "Loan using property",
        "Property backed loan",
        "LAP details",
        "Mortgage finance",
        "Loan on property",
        "Property collateral loan",
        "Property loan eligibility"
    ],

    "general_query": [
        "Tell me about loans",
        "What services do you provide?",
        "Explain banking loans",
        "What loans are available?",
        "Help me understand loans",
        "Loan information",
        "Available loan products",
        "Types of loans",
        "General loan details",
        "Bank loan information"
    ],

    "unknown": [
        "What's the weather today?",
        "Tell me a joke",
        "Who won the cricket match?",
        "Play music",
        "Open YouTube",
        "Book a flight",
        "Translate this sentence",
        "Latest movie releases",
        "News headlines",
        "Order pizza"
    ]
}

with open(INTENT_DATASET_PATH, "w", encoding="utf-8") as f:
    json.dump(intent_dataset, f, indent=4, ensure_ascii=False)

print(f"Intent dataset saved successfully.\n")
print(f"Location: {INTENT_DATASET_PATH}")
print(f"Total intents: {len(intent_dataset)}")
print(f"Total training examples: {sum(len(v) for v in intent_dataset.values())}")

Intent dataset saved successfully.

Location: /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/dataset/intent_dataset.json
Total intents: 19
Total training examples: 235


# Build Intent Dataset

Each intent is assigned a collection of natural user queries covering different ways a person might ask the same question. The resulting dataset is stored as a JSON file and will be used for preprocessing and model training.

In [6]:
# Load intent dataset
with open(INTENT_DATASET_PATH, "r", encoding="utf-8") as f:
    intent_dataset = json.load(f)

# Convert to DataFrame
records = []

for intent, queries in intent_dataset.items():
    for query in queries:
        records.append({
            "query": query,
            "intent": intent
        })

intent_df = pd.DataFrame(records)

print("=" * 60)
print("Dataset Summary")
print("=" * 60)
print(f"Total Samples : {len(intent_df)}")
print(f"Total Intents : {intent_df['intent'].nunique()}")
print()

display(intent_df.head())

print("\nSamples per Intent:\n")
print(intent_df["intent"].value_counts().sort_index())

Dataset Summary
Total Samples : 235
Total Intents : 19



,query,intent
0,Hello,greeting
1,Hi,greeting
2,Good morning,greeting
3,Good evening,greeting
4,Hey,greeting



Samples per Intent:

intent
apply_loan               10
charges_fees             10
education_loan           10
emi_calculation          10
general_query            10
gold_loan                10
greeting                 10
home_loan                10
interest_rate            55
loan_against_property    10
loan_comparison          10
loan_eligibility         10
loan_features            10
personal_loan            10
processing_time          10
repayment                10
required_documents       10
unknown                  10
vehicle_loan             10
Name: count, dtype: int64


# Prepare Text for Training

The classifier works with integer sequences rather than raw text. User queries are cleaned, tokenized, converted into vocabulary indices, and padded to a fixed length so every input has the same shape.

In [7]:
# Basic text preprocessing
def preprocess_text(text):
    text = text.lower().strip()
    text = text.replace("?", "")
    text = text.replace(".", "")
    text = text.replace(",", "")
    return text


intent_df["processed_query"] = intent_df["query"].apply(preprocess_text)

# Tokenization
intent_df["tokens"] = intent_df["processed_query"].apply(str.split)

print(intent_df[["query", "processed_query", "tokens"]].head())

          query processed_query           tokens
0         Hello           hello          [hello]
1            Hi              hi             [hi]
2  Good morning    good morning  [good, morning]
3  Good evening    good evening  [good, evening]
4           Hey             hey            [hey]


# Vocabulary and Sequence Encoding

A vocabulary is created from the training text and each token is mapped to a unique integer. The encoded sequences are then padded to a fixed length, making them suitable for input to the GRU model.

In [8]:
# Build vocabulary
word_counter = Counter()

for tokens in intent_df["tokens"]:
    word_counter.update(tokens)

vocabulary = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in word_counter.items():
    if count >= MIN_WORD_FREQUENCY:
        vocabulary[word] = len(vocabulary)

print(f"Vocabulary Size: {len(vocabulary)}")


def encode_sentence(tokens, vocab, max_len):
    sequence = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

    if len(sequence) < max_len:
        sequence += [vocab["<PAD>"]] * (max_len - len(sequence))
    else:
        sequence = sequence[:max_len]

    return sequence


intent_df["encoded"] = intent_df["tokens"].apply(
    lambda tokens: encode_sentence(
        tokens,
        vocabulary,
        MAX_SEQUENCE_LENGTH
    )
)

# Encode labels
label_encoder = LabelEncoder()
intent_df["label"] = label_encoder.fit_transform(intent_df["intent"])

print("\nEncoded Sample:")
print(intent_df[["query", "encoded", "label"]].head())

Vocabulary Size: 229

Encoded Sample:
          query                                            encoded  label
0         Hello  [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...      6
1            Hi  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...      6
2  Good morning  [4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...      6
3  Good evening  [4, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...      6
4           Hey  [7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...      6


# Split the Dataset

The dataset is divided into training and testing sets using stratified sampling so that each intent is represented in both sets. This provides a fair evaluation of the classifier after training.

In [9]:
# Features and labels
X = intent_df["encoded"].tolist()
y = intent_df["label"].tolist()

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("=" * 50)
print("Dataset Split")
print("=" * 50)
print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

Dataset Split
Training Samples : 188
Testing Samples  : 47


# Create a PyTorch Dataset

A custom dataset converts encoded sequences and labels into tensors. This provides a clean interface for loading batches during training and evaluation.

In [10]:
class IntentDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.sequences[index], self.labels[index]


train_dataset = IntentDataset(X_train, y_train)
test_dataset = IntentDataset(X_test, y_test)

print(f"Training Dataset Size : {len(train_dataset)}")
print(f"Testing Dataset Size  : {len(test_dataset)}")

Training Dataset Size : 188
Testing Dataset Size  : 47


# Prepare DataLoaders

DataLoaders organize the dataset into mini-batches and handle shuffling during training. This improves training efficiency and simplifies the training loop.

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train Batches :", len(train_loader))
print("Test Batches  :", len(test_loader))

Train Batches : 6
Test Batches  : 2


# Build the GRU Classifier

The classifier uses an embedding layer followed by a multi-layer GRU. The final hidden representation is passed through a fully connected layer to predict the intent class.

In [12]:
class GRUIntentClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)

        _, hidden = self.gru(embedded)

        output = self.dropout(hidden[-1])

        return self.fc(output)


model = GRUIntentClassifier(
    vocab_size=len(vocabulary),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=len(label_encoder.classes_),
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

print(model)

GRUIntentClassifier(
  (embedding): Embedding(229, 128, padding_idx=0)
  (gru): GRU(128, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=19, bias=True)
)


# Configure Training

Cross-entropy loss is used for multi-class classification, while the Adam optimizer updates the network parameters during training.

In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print(f"Loss Function : {criterion.__class__.__name__}")
print(f"Optimizer     : {optimizer.__class__.__name__}")

Loss Function : CrossEntropyLoss
Optimizer     : Adam


# Train the Classifier

The model learns by minimizing the classification loss over multiple epochs. Training loss and accuracy are displayed after each epoch to monitor learning progress.

In [14]:
train_losses = []
train_accuracies = []

for epoch in range(NUM_EPOCHS):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS}",
        leave=False
    )

    for inputs, labels in progress_bar:

        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.4f}"
    )

print("\nTraining completed successfully.")

Epoch 1/20:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch [01/20] Loss: 2.8928 | Accuracy: 0.1862


Epoch 2/20:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch [02/20] Loss: 2.8194 | Accuracy: 0.2340


Epoch 3/20:   0%|          | 0/6 [00:00<?, ?it/s]

...

Epoch [18/20] Loss: 2.1970 | Accuracy: 0.2979


Epoch 19/20:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch [19/20] Loss: 2.2274 | Accuracy: 0.2872


Epoch 20/20:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch [20/20] Loss: 2.1701 | Accuracy: 0.2979

Training completed successfully.


# Evaluate the Classifier

The trained model is evaluated on the test dataset. Along with overall accuracy, precision, recall, and F1-score provide a better understanding of the classifier's performance across different intents.

In [15]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for inputs, labels in test_loader:

        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(inputs)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

print("=" * 60)
print("Evaluation Results")
print("=" * 60)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report\n")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)
print("\nConfusion Matrix\n")

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

Evaluation Results
Accuracy : 0.3191
Precision: 0.2483
Recall   : 0.3191
F1 Score : 0.2585

Classification Report

                       precision    recall  f1-score   support

           apply_loan       0.00      0.00      0.00         2
         charges_fees       0.00      0.00      0.00         2
       education_loan       0.18      1.00      0.31         2
      emi_calculation       0.00      0.00      0.00         2
        general_query       0.00      0.00      0.00         2
            gold_loan       0.00      0.00      0.00         2
             greeting       0.00      0.00      0.00         2
            home_loan       0.00      0.00      0.00         2
        interest_rate       0.91      0.91      0.91        11
loan_against_property       0.00      0.00      0.00         2
      loan_comparison       0.00      0.00      0.00         2
     loan_eligibility       0.50      0.50      0.50         2
        loan_features       0.00      0.00      0.00         2
  

# Save Training Artifacts

The trained model and preprocessing objects are saved so they can be loaded directly by the RAG pipeline without retraining.

In [16]:
# Save model weights
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "vocab_size": len(vocabulary),
        "num_classes": len(label_encoder.classes_)
    },
    MODEL_PATH
)
# Save vocabulary
with open(VOCAB_PATH, "wb") as f:
    pickle.dump(vocabulary, f)

# Save label encoder
with open(LABEL_ENCODER_PATH, "wb") as f:
    pickle.dump(label_encoder, f)

# Save tokenizer / preprocessing object
tokenizer_data = {
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "min_word_frequency": MIN_WORD_FREQUENCY
}

with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer_data, f)

print("Artifacts saved successfully.\n")

print(f"Model          : {MODEL_PATH}")
print(f"Vocabulary     : {VOCAB_PATH}")
print(f"Label Encoder  : {LABEL_ENCODER_PATH}")
print(f"Tokenizer      : {TOKENIZER_PATH}")

Artifacts saved successfully.

Model          : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/models/gru_intent_model.pth
Vocabulary     : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/models/vocabulary.pkl
Label Encoder  : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/models/label_encoder.pkl
Tokenizer      : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/models/tokenizer.pkl


# Test Sample Queries

A few unseen queries are passed through the trained model to verify that the prediction pipeline works correctly before integrating it into the chatbot.

In [17]:
def predict_intent(text):

    model.eval()

    processed = preprocess_text(text)

    tokens = processed.split()

    encoded = encode_sentence(
        tokens,
        vocabulary,
        MAX_SEQUENCE_LENGTH
    )

    tensor = torch.tensor(
        [encoded],
        dtype=torch.long
    ).to(DEVICE)

    with torch.no_grad():

        output = model(tensor)

        probabilities = torch.softmax(output, dim=1)

        confidence, prediction = torch.max(probabilities, dim=1)

    return {
        "intent": label_encoder.inverse_transform(
            [prediction.item()]
        )[0],
        "confidence": confidence.item()
    }


sample_queries = [
    "What is the interest rate for home loan?",
    "How can I apply for a personal loan?",
    "Documents required for education loan",
    "Calculate my EMI",
    "Hello",
    "Tell me a joke"
]

print("=" * 60)
print("Sample Predictions")
print("=" * 60)

for query in sample_queries:

    result = predict_intent(query)

    print(f"\nQuery      : {query}")
    print(f"Intent     : {result['intent']}")
    print(f"Confidence : {result['confidence']:.2%}")

Sample Predictions

Query      : What is the interest rate for home loan?
Intent     : interest_rate
Confidence : 99.04%

Query      : How can I apply for a personal loan?
Intent     : loan_eligibility
Confidence : 58.45%

Query      : Documents required for education loan
Intent     : loan_eligibility
Confidence : 26.30%

Query      : Calculate my EMI
Intent     : emi_calculation
Confidence : 6.37%

Query      : Hello
Intent     : education_loan
Confidence : 6.14%

Query      : Tell me a joke
Intent     : emi_calculation
Confidence : 6.16%


# Verify Saved Files

A final check confirms that all required files have been created successfully. These files will be loaded directly by the RAG pipeline in the next notebook.

In [18]:
required_files = {
    "GRU Model": MODEL_PATH,
    "Vocabulary": VOCAB_PATH,
    "Label Encoder": LABEL_ENCODER_PATH,
    "Tokenizer": TOKENIZER_PATH
}

print("=" * 60)
print("Saved Files Verification")
print("=" * 60)

for name, path in required_files.items():
    status = "✓ Found" if os.path.exists(path) else "✗ Missing"
    print(f"{name:<18} : {status}")

print("\nNotebook 4 completed successfully.")

Saved Files Verification
GRU Model          : ✓ Found
Vocabulary         : ✓ Found
Label Encoder      : ✓ Found
Tokenizer          : ✓ Found

Notebook 4 completed successfully.


# Retrieval-Augmented Generation Pipeline

This notebook connects the intent classifier, FAISS vector database, and language model into a single inference pipeline. All models and indexes are loaded from previously generated artifacts, keeping retrieval, intent prediction, and response generation as separate reusable components for later integration with the Streamlit application.

In [ ]:
# ============================# Standard Library Imports# ============================import osimport reimport jsonimport pickleimport warningsfrom pathlib import Pathfrom typing import Dict, List, Tuple# ============================# Data Handling# ============================import numpy as npimport pandas as pd# ============================# PyTorch# ============================import torchimport torch.nn as nnfrom torch.nn.utils.rnn import pad_sequence# ============================# Embedding Model# ============================from sentence_transformers import SentenceTransformer# ============================# Vector Database# ============================import faiss# ============================# Hugging Face Transformers# ============================from transformers import (    AutoTokenizer,    AutoModelForSeq2SeqLM,    pipeline)# ============================# Display Settings# ============================warnings.filterwarnings("ignore")pd.set_option("display.max_columns", None)pd.set_option("display.max_colwidth", 120)# ============================# Device Configuration# ============================DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Using device: {DEVICE}")

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.3 MB/s eta 0:00:00


In [ ]:
# ============================
# Standard Library Imports
# ============================

import os
import re
import json
import pickle
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

# ============================
# Data Handling
# ============================

import numpy as np
import pandas as pd

# ============================
# PyTorch
# ============================

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

# ============================
# Embedding Model
# ============================

from sentence_transformers import SentenceTransformer

# ============================
# Vector Database
# ============================

import faiss

# ============================
# Hugging Face Transformers
# ============================

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

# ============================
# Display Settings
# ============================

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# ============================
# Device Configuration
# ============================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cpu
